In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import tensorflow as tf
tf.keras.utils.set_random_seed(1)

# 1. Read the dataset (Kaggle: Churn_Modelling.csv)
df = pd.read_csv('Churn_Modelling.csv')
df.head()

I0000 00:00:1792818262.039377  110024 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792818262.043514  110024 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792818262.366846  110024 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792818264.981876  110024 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [2]:
# 2. Features / target, drop identifiers, encode categoricals, split
X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
X = pd.get_dummies(X, columns=['Geography', 'Gender'], drop_first=True).astype('float32')
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
# 3. Normalize (fit on train only)
print('BEFORE scaling (train):')
print(X_train.agg(['mean', 'std', 'min', 'max']).T.round(2))

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

print('\nAFTER scaling (train): max |mean| =', abs(X_train.mean(axis=0)).max().round(6),
      '| std range =', X_train.std(axis=0).min().round(3), 'to', X_train.std(axis=0).max().round(3))
print('AFTER scaling (test) : mean =', X_test.mean().round(3), '| std =', X_test.std().round(3),
      '(uses train statistics, so not exactly 0/1)')
print('Scaler fitted on', int(sc.n_samples_seen_), 'train samples only')

BEFORE scaling (train):
                           mean           std     min            max
CreditScore          650.750000     96.599998  350.00     850.000000
Age                   38.950001     10.530000   18.00      92.000000
Tenure                 5.020000      2.900000    0.00      10.000000
Balance            76381.210938  62302.058594    0.00  238387.562500
NumOfProducts          1.530000      0.580000    1.00       4.000000
HasCrCard              0.710000      0.450000    0.00       1.000000
IsActiveMember         0.510000      0.500000    0.00       1.000000
EstimatedSalary    99730.812500  57335.121094   11.58  199992.484375
Geography_Germany      0.250000      0.430000    0.00       1.000000
Geography_Spain        0.250000      0.430000    0.00       1.000000
Gender_Male            0.550000      0.500000    0.00       1.000000

AFTER scaling (train): max |mean| = 0.0 | std range = 1.0 to 1.0
AFTER scaling (test) : mean = -0.006 | std = 1.0 (uses train statistics, so not ex

In [4]:
# 4a. Baseline model
base = Sequential([
    Dense(6, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(6, activation='relu'),
    Dense(1, activation='sigmoid')])
base.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
base.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)

pred = (base.predict(X_test) > 0.5).astype(int)
print('Baseline accuracy:', accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

/home/chinmayee/Chinmayee/College/LP3/LP3B3/venv/lib/python3.14/site-packages/keras/src/layers/core/dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1792818267.559355  110024 cuda_executor.cc:1690] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cuda_status.cc:68

W0000 00:00:1792818267.560015  110171 cuda_executor.cc:1710] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cudnn_api_wr

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  
Baseline accuracy: 0.8655
[[1544   49]
 [ 220  187]]


In [5]:
# 4b. Improved model: wider layers, dropout, early stopping on validation loss
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.2,
          callbacks=[EarlyStopping(patience=25, restore_best_weights=True)], verbose=0)

pred = (model.predict(X_test) > 0.5).astype(int)
print('Improved accuracy:', accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

/home/chinmayee/Chinmayee/College/LP3/LP3B3/venv/lib/python3.14/site-packages/keras/src/layers/core/dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Improved accuracy: 0.865
[[1531   62]
 [ 208  199]]
